[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/recsys-course/03_content_based/03_proyecto_more_like_this.ipynb)

# Proyecto 03 · «Más como esta» para CineMatch
| | |
|---|---|
| **Nivel** | 🟡 Intermedio |
| **Duración** | 3–5 h |
| **GPU** | T4 recomendada (embeddings + CLIP); ≈ 1–2 unidades |
| **Prerrequisitos** | Lección 03 |

## 🏢 Contexto de negocio

Eres ML engineer en **CineMatch**. Producto quiere lanzar la fila **«Más como esta»** en la ficha
de cada película (lo que ves al pulsar un título). Requisitos:

- Debe funcionar **el día del estreno** (sin interacciones): el equipo de contenidos sube sinopsis y póster.
- Debe parecerse a lo que *de verdad* ve la gente que vio la película (los PMs lo comprobarán con datos de co-consumo).
- Latencia: los vecinos se precalculan offline; solo hace falta un *lookup*.

## 📦 Dataset
MovieLens 1M + sinopsis y pósters TMDB vía M³L (mismo loader que la lección, con fallbacks).

## 📋 Entregables y rúbrica

| # | Entregable | Criterio |
|---|---|---|
| 1 | `item_vectors()` que fusiona géneros + texto (+ póster opcional) | Código limpio y documentado |
| 2 | `more_like_this(item_id, k)` | Devuelve títulos coherentes para 5 películas de prueba |
| 3 | **CoWatch-HR@10** (proporción de los 10 vecinos de contenido que están entre los 50 vecinos colaborativos) | **≥ 0,25** (géneros solos ≈ 0,20; reto experto: ≥ 0,35) |
| 4 | **Cold start**: NDCG@10 entre «estrenos» simulados (perfil centrado) | **≥ 2× el aleatorio** (reto: ≥ 3×) |
| 5 | Grid de pósters de 3 consultas + 1 párrafo de análisis de errores | Cualitativo |

Los pesos se eligen en **validación** (ítems de ajuste), y se reportan en ítems de test.

In [ ]:
!pip install -q open_clip_torch sentence-transformers pyarrow

In [ ]:
import os, random, time, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
warnings.filterwarnings("ignore")
seed = 42; random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
device = "cuda" if torch.cuda.is_available() else "cpu"
FAST_DEV_RUN = True
N_POSTERS = 400 if FAST_DEV_RUN else 4000
TEXT_MODEL_KEYS = ["bge-small"] if FAST_DEV_RUN else ["bge-small", "e5-base", "gte-base"]

## 🧰 Utilidades del curso

Para que este notebook sea **autocontenido en Colab**, las dos celdas siguientes
contienen versiones mínimas de las utilidades que construimos en
[01_data](../01_data) (carga de MovieLens + **split temporal global**) y
[02_evaluation](../02_evaluation) (**NDCG@K, Recall@K, coverage**). Son las mismas
definiciones; si ya tienes tu propia librería de los módulos 01/02, puedes
importarla en su lugar.

Convenciones que usaremos en todo el Bloque II:

| Concepto | Convención |
|---|---|
| Columnas | `user_id`, `item_id`, `rating`, `timestamp` (IDs originales de MovieLens) |
| Índices internos | `uidx`, `iidx` contiguos `0..n-1`, definidos **solo con train** |
| Split | temporal **global**: el 80 % más antiguo de las interacciones → train, el 20 % más reciente → test |
| Implícito | positivo = `rating ≥ 4` (protocolo de Liang et al. 2018 / Steck 2019) |
| Evaluación | *full ranking* sobre todo el catálogo, excluyendo lo ya visto en train |

In [ ]:
# 🧰 Utilidades del curso (1/2) — datos. Mismas funciones que en 01_data.
import io, os, zipfile, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse as sp

DATA_DIR = Path(os.environ.get("RECSYS_DATA", "data"))
ML1M_URL = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
# Espejo público en GitHub (mismo contenido, en CSV) por si GroupLens no responde
ML1M_MIRROR = "https://raw.githubusercontent.com/khanhnamle1994/movielens/master"


def load_movielens_1m(data_dir: Path = DATA_DIR):
    """Devuelve (ratings, movies, users) de MovieLens 1M, cacheados en parquet."""
    d = Path(data_dir) / "ml-1m"
    d.mkdir(parents=True, exist_ok=True)
    cache = {n: d / f"{n}.parquet" for n in ("ratings", "movies", "users")}
    if all(p.exists() for p in cache.values()):
        return tuple(pd.read_parquet(p) for p in cache.values())
    try:
        with urllib.request.urlopen(ML1M_URL, timeout=60) as r:
            zipfile.ZipFile(io.BytesIO(r.read())).extractall(d.parent)
        rd = lambda f, cols: pd.read_csv(d / f, sep="::", engine="python", names=cols, encoding="latin-1")
        ratings = rd("ratings.dat", ["user_id", "item_id", "rating", "timestamp"])
        movies = rd("movies.dat", ["item_id", "title", "genres"])
        users = rd("users.dat", ["user_id", "gender", "age", "occupation", "zip"])
    except Exception as e:  # fallback: espejo en GitHub
        print(f"GroupLens no disponible ({e.__class__.__name__}); usando espejo de GitHub…")
        rd = lambda f: pd.read_csv(f"{ML1M_MIRROR}/{f}", sep="\t", index_col=0, encoding="latin-1")
        ratings = rd("ratings.csv").rename(columns={"movie_id": "item_id"})[["user_id", "item_id", "rating", "timestamp"]]
        movies = rd("movies.csv").rename(columns={"movie_id": "item_id"})[["item_id", "title", "genres"]]
        users = rd("users.csv").rename(columns={"zipcode": "zip"})[["user_id", "gender", "age", "occupation", "zip"]]
    movies["genres"] = movies["genres"].str.split("|")
    movies["year"] = movies["title"].str.extract(r"\((\d{4})\)\s*$").astype(float)
    for n, df in zip(cache, (ratings, movies, users)):
        df.to_parquet(cache[n], index=False)
    return ratings, movies, users


def temporal_split(df: pd.DataFrame, test_frac: float = 0.2, ts_col: str = "timestamp"):
    """Split temporal GLOBAL: todo lo anterior al corte → train; lo posterior → test.
    El test se filtra a usuarios e ítems vistos en train (warm-start)."""
    cutoff = df[ts_col].quantile(1 - test_frac)
    train, test = df[df[ts_col] <= cutoff], df[df[ts_col] > cutoff]
    test = test[test.user_id.isin(train.user_id) & test.item_id.isin(train.item_id)]
    return train.reset_index(drop=True), test.reset_index(drop=True)


class Encoder:
    """Mapea IDs originales ↔ índices contiguos (ajustado SOLO con train)."""
    def __init__(self, train: pd.DataFrame):
        self.users = np.sort(train.user_id.unique())
        self.items = np.sort(train.item_id.unique())
        self.u2i = pd.Series(np.arange(len(self.users)), index=self.users)
        self.i2i = pd.Series(np.arange(len(self.items)), index=self.items)
        self.n_users, self.n_items = len(self.users), len(self.items)

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df[df.user_id.isin(self.u2i.index) & df.item_id.isin(self.i2i.index)].copy()
        df["uidx"] = self.u2i.loc[df.user_id].values
        df["iidx"] = self.i2i.loc[df.item_id].values
        return df

    def csr(self, df: pd.DataFrame, values=None) -> sp.csr_matrix:
        """Matriz usuario×ítem dispersa (binaria por defecto)."""
        v = np.ones(len(df), dtype=np.float32) if values is None else np.asarray(values, dtype=np.float32)
        return sp.csr_matrix((v, (df.uidx.values, df.iidx.values)), shape=(self.n_users, self.n_items))

In [ ]:
# 🧰 Utilidades del curso (2/2) — métricas top-K. Mismas definiciones que en 02_evaluation.
from typing import Callable, Dict


def test_relevance(test_enc: pd.DataFrame, min_rating: float = 4.0) -> Dict[int, np.ndarray]:
    """uidx → array de iidx relevantes en test (rating ≥ min_rating)."""
    rel = test_enc[test_enc.rating >= min_rating]
    return {u: g.values for u, g in rel.groupby("uidx")["iidx"]}


def topk_from_scores(scores: np.ndarray, seen: sp.csr_matrix, k: int) -> np.ndarray:
    """Top-K por fila excluyendo ítems ya vistos (scores: batch×n_items, seen: batch×n_items)."""
    scores = np.array(scores, dtype=np.float32, copy=True)
    r, c = seen.nonzero()
    scores[r, c] = -np.inf
    part = np.argpartition(-scores, k, axis=1)[:, :k]
    order = np.take_along_axis(scores, part, 1).argsort(axis=1)[:, ::-1]
    return np.take_along_axis(part, order, 1)


def ndcg_recall_at_k(recs: np.ndarray, relevant: list, k: int):
    """NDCG@K y Recall@K (binarios) para cada usuario."""
    discounts = 1.0 / np.log2(np.arange(2, k + 2))
    ndcg, recall = np.zeros(len(recs)), np.zeros(len(recs))
    for n, (rec, rel) in enumerate(zip(recs, relevant)):
        hits = np.isin(rec[:k], rel)
        idcg = discounts[: min(len(rel), k)].sum()
        ndcg[n] = (hits * discounts).sum() / idcg
        recall[n] = hits.sum() / len(rel)
    return ndcg, recall


def evaluate_topk(score_fn: Callable[[np.ndarray], np.ndarray], train_csr: sp.csr_matrix,
                  relevance: Dict[int, np.ndarray], k: int = 10, batch: int = 512) -> Dict[str, float]:
    """Evalúa un modelo dado por score_fn(uidx_batch) -> scores densos (batch×n_items)."""
    users = np.array(sorted(relevance))
    all_recs = []
    for s in range(0, len(users), batch):
        ub = users[s : s + batch]
        all_recs.append(topk_from_scores(score_fn(ub), train_csr[ub], k))
    recs = np.vstack(all_recs)
    ndcg, rec = ndcg_recall_at_k(recs, [relevance[u] for u in users], k)
    return {f"NDCG@{k}": ndcg.mean(), f"Recall@{k}": rec.mean(),
            "Coverage": len(np.unique(recs)) / train_csr.shape[1], "n_users": len(users)}

### Funciones de la lección (metadatos, embeddings, pósters, CLIP)

In [ ]:
# Metadatos de películas: sinopsis + URL de póster, con cadena de fallbacks
import csv, json, subprocess

M3L_COMMIT = "b11f814041f316684457e43a38cc2a9fd3e9d4eb"  # commit fijado → reproducible
M3L_BASE = f"https://raw.githubusercontent.com/giuspillo/M3L_10M_20M/{M3L_COMMIT}/1_download_raw"


def _from_m3l(d: Path) -> pd.DataFrame:
    """M³L (Spillo et al., 2026): sinopsis TMDB + URL de póster, indexadas por movieId de MovieLens."""
    files = {"REPRO_plot_text.tsv": "download_text", "REPRO_poster_links.tsv": "download_posters"}
    for f, sub in files.items():
        if not (d / f).exists():
            urllib.request.urlretrieve(f"{M3L_BASE}/{sub}/{f}", d / f)
    plots = pd.read_csv(d / "REPRO_plot_text.tsv", sep="\t", quoting=csv.QUOTE_NONE)
    posters = pd.read_csv(d / "REPRO_poster_links.tsv", sep="\t")
    plots = plots.rename(columns={"movie_id": "item_id", "plot": "overview"})
    posters = posters.rename(columns={"movie_id": "item_id", "poster_link": "poster_url"})
    return plots.merge(posters, on="item_id", how="outer")


def _from_kaggle(d: Path) -> pd.DataFrame:
    """Fallback: Kaggle «The Movies Dataset» (requiere ~/.kaggle/kaggle.json)."""
    subprocess.run(["kaggle", "datasets", "download", "-d", "rounakbanik/the-movies-dataset",
                    "-f", "movies_metadata.csv", "-p", str(d), "--unzip"], check=True)
    subprocess.run(["kaggle", "datasets", "download", "-d", "rounakbanik/the-movies-dataset",
                    "-f", "links.csv", "-p", str(d), "--unzip"], check=True)
    meta = pd.read_csv(d / "movies_metadata.csv", low_memory=False)
    meta["tmdbId"] = pd.to_numeric(meta["id"], errors="coerce")
    links = pd.read_csv(d / "links.csv").rename(columns={"movieId": "item_id"})
    out = links.merge(meta[["tmdbId", "overview", "poster_path"]], on="tmdbId", how="left")
    out["poster_url"] = "https://image.tmdb.org/t/p/original" + out["poster_path"].astype(str)
    out.loc[out.poster_path.isna(), "poster_url"] = np.nan
    return out[["item_id", "overview", "poster_url"]]


def load_movie_metadata(movies: pd.DataFrame, data_dir: Path = DATA_DIR) -> pd.DataFrame:
    d = Path(data_dir) / "metadata"
    d.mkdir(parents=True, exist_ok=True)
    meta, source = None, "solo título+géneros (fallback sintético)"
    for name, fn in [("M³L / TMDB (GitHub)", _from_m3l), ("Kaggle The Movies Dataset", _from_kaggle)]:
        try:
            meta, source = fn(d), name
            break
        except Exception as e:
            print(f"⚠️ {name} no disponible: {e.__class__.__name__}")
    out = movies.copy()
    if meta is not None:
        out = out.merge(meta.drop_duplicates("item_id"), on="item_id", how="left")
    for c in ("overview", "poster_url"):
        if c not in out:
            out[c] = np.nan
    out["overview"] = out["overview"].fillna("")
    # Texto del ítem = título + géneros + sinopsis (lo que verá el encoder)
    out["text"] = (out["title"] + ". Genres: " + out["genres"].apply(", ".join) + ". " + out["overview"]).str.strip()
    print(f"Fuente de metadatos: {source} · sinopsis {np.mean(out.overview.str.len() > 0):.1%} · "
          f"pósters {out.poster_url.notna().mean():.1%}")
    return out

In [ ]:
# Embeddings densos de texto con sentence-transformers (fallback: LSA = TF-IDF + SVD truncada)
from sklearn.decomposition import TruncatedSVD

TEXT_MODELS = {  # nombre corto → (modelo en HF Hub, prefijo recomendado por sus autores)
    "bge-small": ("BAAI/bge-small-en-v1.5", ""),
    "e5-base": ("intfloat/e5-base-v2", "query: "),   # E5 exige prefijo; 'query: ' para tareas simétricas
    "gte-base": ("thenlper/gte-base", ""),
}


def embed_texts(texts, model_key: str, cache_dir: Path = DATA_DIR / "emb") -> np.ndarray:
    cache_dir.mkdir(parents=True, exist_ok=True)
    f = cache_dir / f"{model_key}.npy"
    if f.exists():
        return np.load(f)
    name, prefix = TEXT_MODELS[model_key]
    try:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer(name, device=device)
        E = model.encode([prefix + t for t in texts], batch_size=128, normalize_embeddings=True,
                         show_progress_bar=True, convert_to_numpy=True)
    except Exception as e:
        print(f"⚠️ No se pudo cargar {name} ({e.__class__.__name__}); uso LSA como fallback.")
        from sklearn.feature_extraction.text import TfidfVectorizer
        T = TfidfVectorizer(stop_words="english", min_df=2, sublinear_tf=True).fit_transform(texts)
        E = normalize(TruncatedSVD(256, random_state=seed).fit_transform(T))
        f = cache_dir / "lsa_fallback.npy"   # nunca cacheamos el fallback con el nombre del modelo
    E = E.astype(np.float32)
    np.save(f, E)
    return E

In [ ]:
# Descarga paralela de pósters (TMDB sirve tamaños w92…w780; w185 basta para CLIP a 224 px)
from concurrent.futures import ThreadPoolExecutor
from PIL import Image


def download_posters(df: pd.DataFrame, out_dir: Path = DATA_DIR / "posters", size: str = "w185",
                     workers: int = 16) -> dict:
    out_dir.mkdir(parents=True, exist_ok=True)

    def fetch(item_id, url):
        path = out_dir / f"{item_id}.jpg"
        if path.exists() and path.stat().st_size > 0:
            return item_id, path
        try:
            req = urllib.request.Request(url.replace("/original/", f"/{size}/"),
                                         headers={"User-Agent": "Mozilla/5.0 (CineMatch course)"})
            path.write_bytes(urllib.request.urlopen(req, timeout=20).read())
            Image.open(path).verify()
            return item_id, path
        except Exception:
            path.unlink(missing_ok=True)
            return item_id, None

    rows = df.dropna(subset=["poster_url"])
    with ThreadPoolExecutor(workers) as ex:
        res = list(ex.map(lambda r: fetch(r[0], r[1]), zip(rows.item_id, rows.poster_url)))
    return {i: p for i, p in res if p is not None}

In [ ]:
# CLIP (open_clip): imágenes y textos en el MISMO espacio vectorial
CLIP_ARCH, CLIP_WEIGHTS = "ViT-B-32", "laion2b_s34b_b79k"


def load_clip():
    import open_clip
    model, _, preprocess = open_clip.create_model_and_transforms(CLIP_ARCH, pretrained=CLIP_WEIGHTS)
    return model.to(device).eval(), preprocess, open_clip.get_tokenizer(CLIP_ARCH)


@torch.no_grad()
def clip_encode_images(paths, model, preprocess, batch: int = 128) -> np.ndarray:
    out = []
    for s in range(0, len(paths), batch):
        ims = torch.stack([preprocess(Image.open(p).convert("RGB")) for p in paths[s : s + batch]])
        out.append(model.encode_image(ims.to(device)).float().cpu().numpy())
    return normalize(np.vstack(out)).astype(np.float32)


@torch.no_grad()
def clip_encode_texts(texts, model, tokenizer) -> np.ndarray:
    emb = model.encode_text(tokenizer(texts).to(device)).float().cpu().numpy()
    return normalize(emb).astype(np.float32)

In [ ]:
ratings, movies, users = load_movielens_1m()
movies = load_movie_metadata(movies)
train, test = temporal_split(ratings)
row_of = pd.Series(np.arange(len(movies)), index=movies.item_id)

## Paso 1 · La «verdad» de co-consumo (proporcionada)

Para cada película con ≥ 30 *likes* en train calculamos sus 50 vecinos por **coseno colaborativo**
sobre likes (rating ≥ 4). Esa es la referencia con la que los PMs juzgarán la fila. Dividimos esas
películas en **ajuste** (para elegir pesos) y **test** (para reportar).

In [ ]:
likes = train[train.rating >= 4]
enc = Encoder(likes); L = enc.csr(enc.transform(likes))
cnt = np.asarray(L.sum(0)).ravel()
Gm = (L.T @ L).toarray().astype(np.float32); nrm = np.sqrt(np.diag(Gm))
Scf = Gm / np.outer(nrm, nrm).clip(1e-9); np.fill_diagonal(Scf, -1); Scf[:, cnt < 30] = -1
query_iidx = np.where(cnt >= 30)[0]
truth = np.argsort(-Scf[query_iidx], 1)[:, :50]
rng = np.random.default_rng(seed)
is_tune = rng.random(len(query_iidx)) < 0.5
rows_enc = row_of.loc[enc.items].values          # fila de `movies` para cada iidx de `enc`
print(f"{len(query_iidx)} películas consulta · {is_tune.sum()} ajuste / {(~is_tune).sum()} test")

## Paso 2 · Métrica CoWatch-HR@10  ✏️ TODO

Implementa `cowatch_hr(F, mask)` : para cada película consulta (filtrada por `mask`), busca sus 10
vecinos más similares según `F[rows_enc]` (excluyéndose a sí misma y a las películas con `cnt < 30`)
y calcula la fracción que cae dentro de `truth`. Devuelve la media.

💡 Pista: `S = Fq @ F.T`, pon `-inf` en la diagonal y en las columnas no elegibles, `np.argsort(-S, 1)[:, :10]`, `np.isin`.

In [ ]:
def cowatch_hr(F_items: np.ndarray, mask: np.ndarray, k: int = 10) -> float:
    """F_items: matriz (n_items_enc × d) L2-normalizada, alineada con enc.items."""
    # TODO: implementa la métrica
    raise NotImplementedError

## Paso 3 · Representaciones  ✏️ TODO

1. `G`: géneros one-hot normalizados (filas de `movies`).
2. `T`: TF-IDF de `movies.text` (`sublinear_tf=True`, `stop_words="english"`).
3. `E`: embeddings densos con `embed_texts(movies.text.tolist(), "bge-small")`, **centrados** y re-normalizados.
4. (Opcional) `P`: pósters CLIP para las `N_POSTERS` más populares; el resto recibe el póster medio.

Mide `cowatch_hr` en las películas de **ajuste** para cada una.

In [ ]:
from sklearn.preprocessing import normalize, MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
# TODO: construye G, T, E (y opcionalmente P) y mide cowatch_hr(F[rows_enc], is_tune) para cada una
G = T = E = P = None

## Paso 4 · Fusión + año  ✏️ TODO

Implementa `item_vectors(w_genre, w_text, w_img)` (early fusion con $\sqrt{w}$) y un
`more_like_this(item_id, k, gamma)` que reste `gamma · |año_i − año_j| / 10` a la similitud.
Haz un grid sobre `w_genre ∈ {0.2,…,0.8}`, `gamma ∈ {0, 0.05, 0.1, 0.2}` en **ajuste**.

In [ ]:
def item_vectors(w_genre: float, w_text: float, w_img: float = 0.0) -> np.ndarray:
    # TODO
    raise NotImplementedError


def cowatch_hr_year(F_items, mask, gamma, k=10) -> float:
    # TODO: como cowatch_hr pero con penalización por diferencia de año
    raise NotImplementedError

## Paso 5 · Cold start  ✏️ TODO

Reutiliza el protocolo de la lección (15 % de películas con ≥ 20 ratings como «estrenos», borradas
de train; perfil centrado; ranking entre ítems fríos). Compara: aleatorio, géneros, texto, tu fusión.

In [ ]:
# TODO: protocolo de cold start y tabla de resultados

## Paso 6 · Informe  ✏️ TODO
Tabla final en el conjunto de **test** (CoWatch-HR@10 + cold start), grid de pósters de 3 consultas y
un párrafo: ¿en qué falla tu «Más como esta»? (secuelas, remakes, películas infantiles vs adultas…).

---
# ⛔ SPOILER — intenta resolverlo primero

A partir de aquí está la **solución de referencia** completa.

In [ ]:
def _hr_from_sim(S, q, k, truth_rows):
    S[np.arange(len(q)), q] = -np.inf
    S[:, cnt < 30] = -np.inf
    top = np.argsort(-S, 1)[:, :k]
    return float(np.mean([np.isin(t, g).mean() for t, g in zip(top, truth_rows)]))


def cowatch_hr(F_items, mask, k=10):
    q = query_iidx[mask]
    return _hr_from_sim(np.asarray(F_items[q] @ F_items.T, dtype=np.float32), q, k, truth[mask])


years = movies.year.fillna(movies.year.median()).values


def cowatch_hr_year(F_items, mask, gamma, k=10):
    q = query_iidx[mask]
    y = years[rows_enc]
    S = np.asarray(F_items[q] @ F_items.T, dtype=np.float32) - gamma * np.abs(y[q][:, None] - y[None, :]) / 10
    return _hr_from_sim(S, q, k, truth[mask])

In [ ]:
G = normalize(MultiLabelBinarizer().fit_transform(movies.genres).astype(np.float32))
T = TfidfVectorizer(stop_words="english", min_df=2, sublinear_tf=True).fit_transform(movies.text)
T = T.astype(np.float32).toarray()
E_raw = embed_texts(movies.text.tolist(), TEXT_MODEL_KEYS[0])
E = normalize(E_raw - E_raw.mean(0)).astype(np.float32)

# Pósters (opcional): si no hay red a TMDB, P queda en None
pop_like = cnt.copy()
top_ids = enc.items[np.argsort(-pop_like)[:N_POSTERS]]
paths = download_posters(movies[movies.item_id.isin(top_ids)])
P = None
if len(paths) >= 50:
    try:
        cm, cpre, _ = load_clip()
        pids = np.array(sorted(paths))
        Pimg = clip_encode_images([paths[i] for i in pids], cm, cpre)
        P = np.tile(normalize(Pimg.mean(0, keepdims=True)), (len(movies), 1))
        P[row_of.loc[pids].values] = Pimg
    except Exception as e:
        print("CLIP no disponible:", e.__class__.__name__)
single = {"Géneros": G, "TF-IDF": T, TEXT_MODEL_KEYS[0]: E} | ({"Póster CLIP": P} if P is not None else {})
print({k: round(cowatch_hr(v[rows_enc], is_tune), 4) for k, v in single.items()})

In [ ]:
def item_vectors(w_genre, w_text, w_img=0.0):
    blocks = [np.sqrt(w_genre) * G, np.sqrt(w_text) * E]
    if w_img > 0 and P is not None:
        blocks.append(np.sqrt(w_img) * P)
    return normalize(np.hstack(blocks)).astype(np.float32)


grid = {}
for wg in [0.2, 0.4, 0.6, 0.8]:
    for wi in ([0.0, 0.15] if P is not None else [0.0]):
        F = item_vectors(wg, 1 - wg - wi, wi)[rows_enc]
        for gamma in [0.0, 0.05, 0.1, 0.2]:
            grid[(wg, wi, gamma)] = cowatch_hr_year(F, is_tune, gamma)
best = max(grid, key=grid.get)
print("Mejor (w_genre, w_img, gamma) en ajuste:", best, "→", round(grid[best], 4))
gdf = pd.Series(grid).rename_axis(["w_genre", "w_img", "gamma"]).reset_index(name="hr")
fig, ax = plt.subplots(figsize=(7, 3.5))
for (wg, wi), g in gdf.groupby(["w_genre", "w_img"]):
    ax.plot(g.gamma, g.hr, marker="o", label=f"w_genre={wg}, w_img={wi}")
ax.set_xlabel("γ (penalización por año)"); ax.set_ylabel("CoWatch-HR@10 (ajuste)"); ax.legend(fontsize=7)
ax.set_title("Efecto de la fusión y del año"); plt.show()

In [ ]:
wg, wi, gamma = best
F_best_all = item_vectors(wg, 1 - wg - wi, wi)


def more_like_this(item_id: int, k: int = 10, gamma: float = gamma):
    i = row_of[item_id]
    s = F_best_all @ F_best_all[i] - gamma * np.abs(years - years[i]) / 10
    s[i] = -np.inf
    top = np.argsort(-s)[:k]
    return movies.iloc[top][["item_id", "title"]].assign(score=s[top].round(3))


final = {k: cowatch_hr(v[rows_enc], ~is_tune) for k, v in single.items()}
final["Fusión + año (CineMatch)"] = cowatch_hr_year(F_best_all[rows_enc], ~is_tune, gamma)
final["Aleatorio"] = float(np.mean([np.isin(rng.choice(np.where(cnt >= 30)[0], 10, replace=False), g).mean() for g in truth[~is_tune]]))
print(pd.Series(final).sort_values().round(4))
for mid in [1, 260, 1196]:
    print("\n🎬", movies.title[row_of[mid]]); print(more_like_this(mid, 5).to_string(index=False))

In [ ]:
# Cold start (mismo protocolo que la lección)
import scipy.sparse as sp
cnt_all = ratings.item_id.value_counts()
eligible = cnt_all[cnt_all >= 20].index.values
cold = np.sort(np.random.default_rng(seed).choice(eligible, size=int(0.15 * len(eligible)), replace=False))
tr_w = train[~train.item_id.isin(cold)]
enc_w = Encoder(tr_w); tw = enc_w.transform(tr_w)
Rw = enc_w.csr(tw, tw.rating.values).tocsr()
cts = np.diff(Rw.indptr); Rw.data -= np.repeat(np.asarray(Rw.sum(1)).ravel() / np.maximum(cts, 1), cts)
cpos = pd.Series(np.arange(len(cold)), index=cold)
lc = ratings[ratings.item_id.isin(cold) & ratings.user_id.isin(enc_w.users) & (ratings.rating >= 4)]
rel_cold = {u: g.values for u, g in lc.assign(u=enc_w.u2i.loc[lc.user_id].values, c=cpos.loc[lc.item_id].values).groupby("u")["c"]}
no_seen = sp.csr_matrix((enc_w.n_users, len(cold)))
wr, cr = row_of.loc[enc_w.items].values, row_of.loc[cold].values
cold_eval = lambda F: evaluate_topk(lambda ub: (Rw[ub] @ F[wr]) @ F[cr].T, no_seen, rel_cold)["NDCG@10"]
cold_res = {"Aleatorio": evaluate_topk(lambda ub: rng.random((len(ub), len(cold))), no_seen, rel_cold)["NDCG@10"],
            "Géneros": cold_eval(G), TEXT_MODEL_KEYS[0]: cold_eval(E), "Fusión CineMatch": cold_eval(F_best_all)}
print(pd.Series(cold_res).round(4)); print("ratio vs aleatorio:", round(cold_res["Fusión CineMatch"] / cold_res["Aleatorio"], 2))

In [ ]:
# Grid cualitativo de pósters
from PIL import Image
if P is not None:
    for mid in [1, 2571, 1258]:
        if mid not in paths: continue
        recs = [r for r in more_like_this(mid, 30).item_id if r in paths][:5]
        fig, axes = plt.subplots(1, 6, figsize=(12, 3.2))
        for ax, i in zip(axes, [mid] + recs):
            ax.imshow(Image.open(paths[i])); ax.axis("off"); ax.set_title(movies.title[row_of[i]][:18], fontsize=7)
        plt.suptitle("Consulta (izq.) → «Más como esta»"); plt.show()
else:
    print("Sin pósters disponibles: omito el grid.")

**Análisis de errores (solución de referencia)**: la fusión acierta franquicias y subgéneros, pero
(1) mezcla películas infantiles con adultas que comparten género *Animation*; (2) los remakes salen
arriba aunque su público sea otro; (3) el año ayuda mucho porque los públicos de MovieLens están
muy cohortizados por época; (4) **la mejor fusión para co-consumo (mucho peso en géneros) no es la
mejor para cold start (donde gana el texto)**: objetivos distintos piden pesos distintos, así que en
producción se mantienen dos configuraciones (fila «Más como esta» vs. candidatos de estrenos). La señal
colaborativa (módulo 04) corrige casi todo esto en ítems warm.

## 🚀 Retos extra (nivel experto)

1. **Alinear contenido con CF**: entrena una regresión (ridge o MLP en PyTorch) que prediga el
   vector colaborativo de un ítem (fila de `Scf` reducida con SVD) desde `[G, E, P]`. Úsala para el
   cold start y compárala con la fusión (idea de van den Oord 2013 / DropoutNet).
2. **Fine-tuning contrastivo** de `bge-small` con pares (ítem, vecino colaborativo) usando
   `MultipleNegativesRankingLoss` de sentence-transformers. ¿Sube CoWatch-HR sin romper el cold start?
3. **Sinopsis en español** con `intfloat/multilingual-e5-base` traduciendo un subconjunto con un LLM.
4. **Índice ANN**: indexa `F_best_all` en FAISS y mide la latencia de `more_like_this` (anticipo del módulo 08).

## 🤔 Reflexión (producción / MLOps)

- ¿Cómo versionarías los embeddings para que un cambio de encoder no rompa la fila en producción?
- ¿Qué monitorizarías tras el lanzamiento (CTR de la fila, cobertura de estrenos, drift del encoder)?
- ¿Cuándo «graduarías» un estreno del contenido al colaborativo? ¿Con cuántas interacciones?